# Echo (EchoNet-Dynamic) — ejection fraction regression (Google Colab)

Colab port of [`notebooks/kaggle/04_echo_ef_training.ipynb`](../kaggle/04_echo_ef_training.ipynb).
**Re-check Hugging Face for an Echo-Vision-FM-class checkpoint before running
this** — per `docs/04-data-models.md`, none was verified at planning time, but
that's the kind of thing that changes.

Like BraTS, EchoNet-Dynamic has no simple direct-download URL — the official
release requires a Stanford data-use agreement
([echonet.github.io/dynamic](https://echonet.github.io/dynamic/)). This
notebook supports the same two acquisition paths as the BraTS Colab notebook:
Kaggle API (for prototyping against an unofficial mirror) or your own copy in
Drive (recommended once you've gone through the official DUA).

## Colab setup
1. **Runtime → Change runtime type → T4 GPU.**
2. Mount Drive (cell 2) for the trained checkpoint.
3. Acquire the dataset via Option A or Option B below.

In [ ]:
# Cell 1 — install packages.
!pip install -q opencv-python-headless

In [ ]:
# Cell 2 — mount Drive for the trained checkpoint.
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_WORKDIR = "/content/drive/MyDrive/aegis-dx/echo-ef"
os.makedirs(DRIVE_WORKDIR, exist_ok=True)

## Option A — Kaggle API (unofficial mirror, fine for prototyping)

Same process as the BraTS Colab notebook: create a Kaggle API token at
[kaggle.com/settings](https://www.kaggle.com/settings), upload it below, then
search *kaggle.com/datasets* for **"EchoNet-Dynamic"** and copy the exact API
command from the dataset page into `KAGGLE_DATASET_SLUG`.

In [ ]:
# Cell 3 — Option A: Kaggle API download. Skip if using Option B (Drive) instead.
from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select your kaggle.json when prompted
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

KAGGLE_DATASET_SLUG = "REPLACE_WITH_SLUG_FROM_KAGGLE_DATASET_PAGE"  # fill this in - see markdown above

!pip install -q kaggle
!kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p /content/echonet_download --unzip
ECHONET_ROOT = "/content/echonet_download"
print("Downloaded to:", ECHONET_ROOT)

## Option B — official data, already in Drive

**Recommended for anything beyond prototyping** — go through Stanford's
official DUA, download `FileList.csv` + `Videos/`, upload once to Drive, then
point this cell at that folder instead of running cell 3.

In [ ]:
# Cell 4 — Option B: point at EchoNet-Dynamic already sitting in Drive.
# ECHONET_ROOT = "/content/drive/MyDrive/aegis-dx/echonet-dynamic-official"
print("ECHONET_ROOT is set to:", ECHONET_ROOT)

In [ ]:
# Cell 5 — imports and reproducibility.
import glob
import json
import random

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected - check Runtime > Change runtime type > T4 GPU.")

In [ ]:
# Cell 6 — locate FileList.csv and Videos/ under ECHONET_ROOT (handles an
# extra nesting level some mirrors add).
file_list_candidates = glob.glob(f"{ECHONET_ROOT}/**/FileList.csv", recursive=True)
if not file_list_candidates:
    raise FileNotFoundError(f"No FileList.csv found under {ECHONET_ROOT}. Check cell 3 or 4 above.")
ECHONET_ROOT = os.path.dirname(file_list_candidates[0])

VIDEOS_DIR = os.path.join(ECHONET_ROOT, "Videos")
if not os.path.isdir(VIDEOS_DIR):
    nested = glob.glob(os.path.join(ECHONET_ROOT, "**", "Videos"), recursive=True)
    VIDEOS_DIR = nested[0] if nested else VIDEOS_DIR
print("EchoNet root:", ECHONET_ROOT)
print("Videos dir:", VIDEOS_DIR)

In [ ]:
# Cell 7 — load metadata and the official train/val/test split (use as-is).
file_list = pd.read_csv(os.path.join(ECHONET_ROOT, "FileList.csv"))
print(file_list[["FileName", "EF", "Split"]].head())
print(file_list.Split.value_counts())

train_df = file_list[file_list.Split == "TRAIN"].reset_index(drop=True)
val_df = file_list[file_list.Split == "VAL"].reset_index(drop=True)
test_df = file_list[file_list.Split == "TEST"].reset_index(drop=True)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

In [ ]:
# Cell 8 — Dataset class: frame-sampled clips, identical to the Kaggle version.
NUM_FRAMES = 16
FRAME_SIZE = 112


def read_sampled_frames(video_path: str, num_frames: int, frame_size: int) -> np.ndarray:
    capture = cv2.VideoCapture(video_path)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames).astype(int)

    frames = []
    for target_index in frame_indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(target_index))
        success, frame = capture.read()
        if not success:
            frame = np.zeros((frame_size, frame_size, 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, (frame_size, frame_size))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    capture.release()

    clip = np.stack(frames).astype("float32") / 255.0
    clip = (clip - 0.5) / 0.5
    return clip.transpose(3, 0, 1, 2)


class EchoNetDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, videos_dir: str):
        self.dataframe = dataframe
        self.videos_dir = videos_dir

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        filename = row.FileName if row.FileName.endswith(".avi") else f"{row.FileName}.avi"
        video_path = os.path.join(self.videos_dir, filename)
        clip = read_sampled_frames(video_path, NUM_FRAMES, FRAME_SIZE)
        ejection_fraction = np.float32(row.EF) / 100.0
        return torch.from_numpy(clip), torch.tensor(ejection_fraction)


train_ds = EchoNetDataset(train_df, VIDEOS_DIR)
val_ds = EchoNetDataset(val_df, VIDEOS_DIR)
test_ds = EchoNetDataset(test_df, VIDEOS_DIR)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# Cell 9 — model: per-frame CNN encoder + temporal attention pooling (identical to Kaggle).
class EchoEFRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        self.frame_encoder = backbone
        self.frame_feature_dim = 512

        self.temporal_attention = nn.Sequential(
            nn.Linear(self.frame_feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.regression_head = nn.Sequential(
            nn.Linear(self.frame_feature_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        batch_size, channels, num_frames, height, width = clip.shape
        frames = clip.permute(0, 2, 1, 3, 4).reshape(batch_size * num_frames, channels, height, width)
        frame_features = self.frame_encoder(frames).view(batch_size, num_frames, self.frame_feature_dim)

        attention_logits = self.temporal_attention(frame_features).squeeze(-1)
        attention_weights = torch.softmax(attention_logits, dim=1).unsqueeze(-1)
        pooled_features = (attention_weights * frame_features).sum(dim=1)

        return self.regression_head(pooled_features).squeeze(-1)


model = EchoEFRegressor().to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

In [ ]:
# Cell 10 — training loop, checkpointing best-so-far to Drive.
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
CHECKPOINT_PATH = os.path.join(DRIVE_WORKDIR, "echonet_ef_regressor_best.pt")


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    all_true, all_pred = [], []
    for clips, ef_true in loader:
        clips = clips.to(DEVICE)
        ef_pred = model(clips).cpu()
        all_true.append(ef_true * 100.0)
        all_pred.append(ef_pred * 100.0)
    y_true = torch.cat(all_true).numpy()
    y_pred = torch.cat(all_pred).numpy()
    return mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred)


NUM_EPOCHS = 10
best_val_mae = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for clips, ef_true in train_loader:
        clips, ef_true = clips.to(DEVICE), ef_true.to(DEVICE)
        optimizer.zero_grad()
        ef_pred = model(clips)
        loss = criterion(ef_pred, ef_true)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * clips.size(0)

    train_loss = running_loss / len(train_ds)
    val_mae, val_r2 = evaluate(val_loader)
    print(f"epoch {epoch+1:02d}  train_mae_norm={train_loss:.4f}  val_EF_MAE={val_mae:.2f}  val_R2={val_r2:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), CHECKPOINT_PATH)

model.load_state_dict(torch.load(CHECKPOINT_PATH))
print("Best val EF MAE:", best_val_mae)

In [ ]:
# Cell 11 — held-out test evaluation + save metadata sidecar.
test_mae, test_r2 = evaluate(test_loader)
print(f"Test EF MAE: {test_mae:.2f} percentage points")
print(f"Test R^2: {test_r2:.4f}")

metadata = {
    "model": "EchoEFRegressor",
    "num_frames_sampled": NUM_FRAMES,
    "frame_size": FRAME_SIZE,
    "test_ef_mae": float(test_mae),
    "test_r2": float(test_r2),
    "trained_on": "EchoNet-Dynamic, official FileList.csv Split column",
    "trained_on_platform": "Google Colab",
}
with open(os.path.join(DRIVE_WORKDIR, "echonet_ef_regressor_best.json"), "w") as handle:
    json.dump(metadata, handle, indent=2)
print(json.dumps(metadata, indent=2))

## Next steps

Same as the Kaggle version: re-check for an Echo-Vision-FM checkpoint before
committing further to this path, add the wall-motion classification head
(not built here — EF regression only), subgroup breakdown, calibration,
registry entry. If you prototyped against an unofficial Kaggle mirror,
switch to the official Stanford release before anything beyond internal
experimentation. See `notebooks/kaggle/04_echo_ef_training.ipynb`'s final cell.